# Pure Functional Randomness in JAX

This notebook explores JAX's unique approach to randomness using explicit PRNG keys instead of global mutable state.

## Why Randomness Is Special in JAX

### 🧠 The Core Difference

**NumPy approach:**
```python
np.random.rand()  # → 0.123
np.random.rand()  # → 0.456
```
Each call gives new random numbers because NumPy uses **global mutable state**.

**JAX approach:**
- **No hidden state** allowed
- **Explicit randomness** via PRNG keys
- **You control** randomness manually

This makes JAX:
- **JIT safe** - compiler can optimize
- **Parallel safe** - no race conditions
- **Reproducible** - same key = same result

In [1]:
# Import JAX libraries
import jax
import jax.numpy as jnp

### ✅ Exercise 21 — Create a PRNG Key

Because JAX is functional, instead of asking for randomness, you provide a key and ask JAX to use it.

In [2]:
# PRNG keys are just arrays containing state
# Think of them as "seeds" that get consumed when used

key = jax.random.PRNGKey(100) # This key is just an array, it contains state
key

Array([  0, 100], dtype=uint32)

### ✅ Exercise 22 — Split the Key

Keys are single-use only. After using a key, you must split it to get new keys for future operations.

In [3]:
# Split creates a new key + subkey
# Rule: Never reuse keys - always split first!
key, subkey = jax.random.split(key)
key, subkey

(Array([ 701046466, 2104227382], dtype=uint32),
 Array([2213033797, 2583756506], dtype=uint32))

### ✅ Exercise 23 — Split Into Many Keys

For multiple random operations, you can split a key into multiple subkeys at once.

In [4]:
# One key per random operation
# Each key gets consumed when used
key_1, key_2, key_3, key_4, key_5, key_6, key_7 = jax.random.split(key, num=7)
key_1, key_2, key_3, key_4, key_5, key_6, key_7

(Array([3549270403, 1145628597], dtype=uint32),
 Array([1315821631, 2840677502], dtype=uint32),
 Array([1953597499, 2148997509], dtype=uint32),
 Array([1420903369,  369168748], dtype=uint32),
 Array([ 242814072, 1159027379], dtype=uint32),
 Array([ 194610812, 2464920650], dtype=uint32),
 Array([3641069794,   35145164], dtype=uint32))

### ✅ Exercise 24 — Random Permutation

Shuffle the order of array elements using a specific PRNG key.

In [5]:
sample_data = jnp.array([10, 1, 24, 20, 15, 14])
data_permutation = jax.random.permutation(key_1, sample_data)
data_permutation

Array([15, 20, 10, 14,  1, 24], dtype=int32)

### ✅ Exercise 25 — Random Choice

Select a random element from an array using the specified PRNG key.

In [6]:
random_selection = jax.random.choice(key_2, sample_data)
random_selection

Array(15, dtype=int32)

### ✅ Exercise 26 — Random Integer

Generate random integers within a specified range [minval, maxval).

In [7]:
sample_int = jax.random.randint(
    key_3,
    shape=(1,),      # output shape
    minval=10,       # inclusive minimum
    maxval=24        # exclusive maximum
)
sample_int

Array([20], dtype=int32)

### ✅ Exercise 27 — Uniform Distribution

Generate random numbers from a uniform distribution where all values in the range have equal probability.

In [8]:
# Equal probability between 1 and 2
sample_uniform = jax.random.uniform(
    key_4,
    shape=(2,),
    minval=1,
    maxval=2
)
sample_uniform

Array([1.2547921, 1.2823569], dtype=float32)

### ✅ Exercise 28 — Bernoulli Distribution

Generate random boolean values - like coin flips with a specified probability.

In [9]:
# Bernoulli = coin flip (True/False)
sample_bernoulli = jax.random.bernoulli(
    key_5,
    shape=(3,)
)
sample_bernoulli

Array([ True,  True,  True], dtype=bool)

### ✅ Exercise 29 — Poisson Distribution

Generate random counts from a Poisson distribution, useful for modeling events occurring at a constant rate.

In [10]:
# lam = expected count (lambda parameter)
sample_poisson = jax.random.poisson(
    key_6,
    shape=(2, 3),
    lam=100
)
sample_poisson

W0304 12:54:24.959561 10375394 cpp_gen_intrinsics.cc:74] Empty bitcode string provided for eigen. Optimizations relying on this IR will be disabled.


Array([[ 94, 109, 106],
       [ 89, 100,  84]], dtype=int32)

### ✅ Exercise 30 — Normal Distribution

Generate random numbers from a normal (Gaussian) distribution with mean 0 and standard deviation 1.

In [11]:
# Standard normal distribution (mean=0, std=1)
sample_normal = jax.random.normal(
    key_7,
    shape=(2, 3, 4)
)
sample_normal

Array([[[ 0.34660995,  0.02455962,  1.0251855 ,  0.763851  ],
        [-2.0041497 ,  0.1554214 , -1.7528274 ,  2.0635715 ],
        [-0.17306615,  1.4471363 ,  0.05802692, -0.19730496]],

       [[ 0.31210348,  0.67494154, -0.5323124 ,  0.08658653],
        [-0.8526911 ,  0.03244016,  0.29185736,  1.2332078 ],
        [ 1.6374227 ,  0.33908826, -1.8585606 ,  0.7492276 ]]],      dtype=float32)

## Conclusion

### The JAX Philosophy of Randomness

The core theme is **explicit, functional randomness**.

**Traditional approach:**
```python
"Give me randomness"  # Implicit, hidden state
```

**JAX approach:**
```python
"Here is a key. Transform it."  # Explicit, functional
```

This design is:
- **Deterministic** - Same key always produces same result
- **Reproducible** - Easy to share and reproduce experiments
- **Parallel safe** - No race conditions from shared state
- **JIT safe** - Compiler can optimize effectively

By making randomness explicit, JAX enables powerful optimizations while maintaining correctness.